In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs into DataFrames (stored for later steps)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error_logs = pd.read_csv('error_logs.csv')

# Helper: convert timestamp (seconds) to timezone-aware datetime in Asia/Shanghai (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
for df in (df_metric, df_trace, df_log, df_error_logs):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz_sh)

# 1) metric.csv summaries
metric_cmdb_counts = df_metric['cmdb_id'].value_counts().reset_index().rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'}).head(20)
metric_kpi_counts = df_metric['kpi_name'].value_counts().reset_index().rename(columns={'index': 'kpi_name', 'kpi_name': 'count'}).head(20)
metric_time_range = pd.DataFrame({
    'min_timestamp': [df_metric['dt'].min()],
    'max_timestamp': [df_metric['dt'].max()]
})

# 2) trace.csv summaries
trace_cmdb_counts = df_trace['cmdb_id'].value_counts().reset_index().rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'}).head(20)
trace_feature_counts = df_trace['trace_name'].value_counts().reset_index().rename(columns={'index': 'trace_name', 'trace_name': 'count'}).head(20)
trace_time_range = pd.DataFrame({
    'min_timestamp': [df_trace['dt'].min()],
    'max_timestamp': [df_trace['dt'].max()]
})

# 3) log.csv summaries
log_cmdb_counts = df_log['cmdb_id'].value_counts().reset_index().rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'}).head(20)
log_feature_counts = df_log['log_name'].value_counts().reset_index().rename(columns={'index': 'log_name', 'log_name': 'count'}).head(20)
log_time_range = pd.DataFrame({
    'min_timestamp': [df_log['dt'].min()],
    'max_timestamp': [df_log['dt'].max()]
})

# 4) error_logs.csv summaries
error_cmdb_counts = df_error_logs['cmdb_id'].value_counts().reset_index().rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'}).head(20)
# distinct message sample count and top message samples
error_message_distinct_count = df_error_logs['message'].nunique()
error_message_counts = df_error_logs['message'].value_counts().reset_index().rename(columns={'index': 'message', 'message': 'count'}).head(20)
error_time_range = pd.DataFrame({
    'min_timestamp': [df_error_logs['dt'].min()],
    'max_timestamp': [df_error_logs['dt'].max()]
})

# Compact output (top 20 per list, and min/max timestamps). Variables below are displayed as the result.
metric_cmdb_counts, metric_kpi_counts, metric_time_range, trace_cmdb_counts, trace_feature_counts, trace_time_range, log_cmdb_counts, log_feature_counts, log_time_range, error_cmdb_counts, error_message_distinct_count, error_message_counts, error_time_range

```
Out[1]:
```
Summary of the loaded telemetry files (compact, top items only):

metric.csv
- Top cmdb_id by row count (top 20 shown): Mysql01 4886, Mysql02 4807, Tomcat03 2037, Tomcat04 2025, Tomcat02 2022, Tomcat01 1889, Redis02 1840, MG01 1639, MG02 1635, Redis01 1543, IG02 1094, IG01 1078, apache02 926, apache01 868, ServiceTest10 124, ServiceTest1 124, ServiceTest11 124, ServiceTest2 124, ServiceTest8 124, ServiceTest7 124.
- Top KPI names (kpi_name) by count (top 20): OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 352, OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT 352, OSLinux-OSLinux_ZABBIX_Host_Uptime 347, OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies 347, OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 347, app.mrt 341, app.rr 341, app.sr 341, app.cnt 341, OSLinux-CPU_CPU_CPULoad 338, OSLinux-CPU_CPU_CPUidleutil 338, OSLinux-CPU_CPU_CPUUserTime 336, OSLinux-CPU_CPU_CPUSysTime 336, ... (see top 20).
- Timestamp range (UTC+8): 2021-03-25 19:00:00+08:00 to 2021-03-25 19:30:00+08:00.

trace.csv
- Top cmdb_id by row count (top 20 shown): MG01 1200, MG02 1188, dockerB1 471, IG02 468, Tomcat04 468, Tomcat01 468, IG01 465, Tomcat02 465, Tomcat03 465, dockerB2 462, dockerA2 453, dockerA1 453, ...
- Top trace features (trace_name) by count (top 20): trace.self.duration_mean 370, trace.self.duration_p95 370, trace.self.row_count 370, trace.to_MG01.duration_mean 245, trace.to_MG01.duration_p95 245, trace.to_MG01.row_count 245, trace.to_MG02.duration_mean 242, ... (top 20).
- Timestamp range (UTC+8): 2021-03-25 19:00:00+08:00 to 2021-03-25 19:30:00+08:00.

log.csv
- Top cmdb_id by row count (top 20 shown): Tomcat04 73, Tomcat02 72, Tomcat03 72, Tomcat01 71, apache01 62, apache02 62, Mysql01 2, Mysql02 2.
- Top log features (log_name) by count: log.error_count 188, log.row_count 188, log.gc_pause_max 40.
- Timestamp range (UTC+8): 2021-03-25 19:00:00+08:00 to 2021-03-25 19:30:00+08:00.

error_logs.csv
- Top cmdb_id by row count (top 20 shown): apache01 70, apache02 59.
- Distinct error message count (sampled): 129 distinct messages.
- Top message samples (top 20 shown): many unique HTTP POST messages (each appears once in the top list), e.g., entries like 'IPAddress "POST /UOCP/person/ServiceTest10.json ...' (each count = 1).
- Timestamp range (UTC+8): 2021-03-25 19:15:04+08:00 to 2021-03-25 19:19:14+08:00.

If you want, I can: (a) show more items (beyond top 20) for any list, (c) extract specific KPI/trace/log series for a given cmdb_id and time window, or (d) compute simple aggregates (min/max/mean) per KPI.

The original code execution output of IPython Kernel is also provided below for reference:

(            count  count
0         Mysql01   4886
1         Mysql02   4807
2        Tomcat03   2037
3        Tomcat04   2025
4        Tomcat02   2022
5        Tomcat01   1889
6         Redis02   1840
7            MG01   1639
8            MG02   1635
9         Redis01   1543
10           IG02   1094
11           IG01   1078
12       apache02    926
13       apache01    868
14  ServiceTest10    124
15   ServiceTest1    124
16  ServiceTest11    124
17   ServiceTest2    124
18   ServiceTest8    124
19   ServiceTest7    124,                                                 count  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    352
1        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    352
2                  OSLinux-OSLinux_ZABBIX_Host_Uptime    347
3       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    347
4          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    347
5                                             app.mrt    341
6                                              app.rr    341
7                                              app.sr    341
8                                             app.cnt    341
9                             OSLinux-CPU_CPU_CPULoad    338
10                        OSLinux-CPU_CPU_CPUidleutil    338
11                        OSLinux-CPU_CPU_CPUUserTime    336
12                         OSLinux-CPU_CPU_CPUSysTime    336
13  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    336
14                             OSLinux-CPU_CPU_CPUWio    336
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    291
16       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    278
17                         OSLinux-CPU_CPU_CPUCpuUtil    278
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    268
19             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem    260,               min_timestamp             max_timestamp
0 2021-03-25 19:00:00+08:00 2021-03-25 19:30:00+08:00,        count  count
0       MG01   1200
1       MG02   1188
2   dockerB1    471
3       IG02    468
4   Tomcat04    468
5   Tomcat01    468
6       IG01    465
7   Tomcat02    465
8   Tomcat03    465
9   dockerB2    462
10  dockerA2    453
11  dockerA1    453,                             count  count
0        trace.self.duration_mean    370
1         trace.self.duration_p95    370
2            trace.self.row_count    370
3     trace.to_MG01.duration_mean    245
4      trace.to_MG01.duration_p95    245
5         trace.to_MG01.row_count    245
6     trace.to_MG02.duration_mean    242
7         trace.to_MG02.row_count    242
8      trace.to_MG02.duration_p95    242
9   trace.from_IG02.duration_mean    123
10   trace.from_IG02.duration_p95    123
11      trace.from_IG02.row_count    123
12   trace.from_IG01.duration_p95    122
13  trace.from_IG01.duration_mean    122
14  trace.from_MG01.duration_mean    122
15      trace.from_MG01.row_count    122
16   trace.from_MG01.duration_p95    122
17      trace.from_IG01.row_count    122
18      trace.from_MG02.row_count    121
19  trace.from_MG02.duration_mean    121,               min_timestamp             max_timestamp
0 2021-03-25 19:00:00+08:00 2021-03-25 19:30:00+08:00,       count  count
0  Tomcat04     73
1  Tomcat02     72
2  Tomcat03     72
3  Tomcat01     71
4  apache01     62
5  apache02     62
6   Mysql01      2
7   Mysql02      2,               count  count
0   log.error_count    188
1     log.row_count    188
2  log.gc_pause_max     40,               min_timestamp             max_timestamp
0 2021-03-25 19:00:00+08:00 2021-03-25 19:30:00+08:00,       count  count
0  apache01     70
1  apache02     59, 129,                                                 count  count
0   IPAddress "POST /UOCP/person/ServiceTest10.jso...      1
1   IPAddress "POST /UOCP/person/ServiceTest1.json...      1
2   IPAddress "POST /UOCP/person/ServiceTest9.json...      1
3   IPAddress "POST /UOCP/base/ServiceTest7.json H...      1
4   IPAddress "POST /UOCP/person/ServiceTest1.json...      1
5   IPAddress "POST /UOCP/base/ServiceTest8.json H...      1
6   IPAddress "POST /UOCP/base/ServiceTest2.json H...      1
7   IPAddress "POST /UOCP/person/ServiceTest1.json...      1
8   IPAddress "POST /UOCP/person/ServiceTest5.json...      1
9   IPAddress "POST /UOCP/person/ServiceTest10.jso...      1
10  IPAddress "POST /UOCP/base/ServiceTest11.json ...      1
11  IPAddress "POST /UOCP/person/ServiceTest10.jso...      1
12  IPAddress "POST /UOCP/person/ServiceTest10.jso...      1
13  IPAddress "POST /UOCP/base/ServiceTest4.json H...      1
14  IPAddress "POST /UOCP/base/ServiceTest6.json H...      1
15  IPAddress "POST /UOCP/person/ServiceTest1.json...      1
16  IPAddress "POST /UOCP/person/ServiceTest5.json...      1
17  IPAddress "POST /UOCP/base/ServiceTest4.json H...      1
18  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1
19  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1,               min_timestamp             max_timestamp
0 2021-03-25 19:15:04+08:00 2021-03-25 19:19:14+08:00)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse previously loaded DataFrames: df_metric, df_trace, df_log, df_error_logs
# Ensure timezone-aware dt columns exist (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
for df in (df_metric, df_trace, df_log, df_error_logs):
    if 'timestamp' in df.columns and 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz_sh)

# Allowed components list
allowed_components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# 1) metric.csv: group by cmdb_id and kpi_name -> count and global P95
# Filter to allowed components
df_metric_allowed = df_metric[df_metric['cmdb_id'].isin(allowed_components)].copy()

# Ensure 'value' is numeric
df_metric_allowed['value'] = pd.to_numeric(df_metric_allowed['value'], errors='coerce')

agg_metric = (
    df_metric_allowed
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
)

# Sort by count desc and keep top 20
agg_metric_top20 = agg_metric.sort_values('count', ascending=False).head(20)

# Allowed components with zero metric rows
present_metric_components = df_metric_allowed['cmdb_id'].unique().tolist()
zero_metric_components = sorted([c for c in allowed_components if c not in present_metric_components])

# 2) trace.csv: group by cmdb_id and trace_name -> count and global P95
df_trace_allowed = df_trace[df_trace['cmdb_id'].isin(allowed_components)].copy()
df_trace_allowed['value'] = pd.to_numeric(df_trace_allowed['value'], errors='coerce')

agg_trace = (
    df_trace_allowed
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
)

agg_trace_top20 = agg_trace.sort_values('count', ascending=False).head(20)

present_trace_components = df_trace_allowed['cmdb_id'].unique().tolist()
zero_trace_components = sorted([c for c in allowed_components if c not in present_trace_components])

# 3) log.csv: group by cmdb_id and log_name -> count and global P95
df_log_allowed = df_log[df_log['cmdb_id'].isin(allowed_components)].copy()
df_log_allowed['value'] = pd.to_numeric(df_log_allowed['value'], errors='coerce')

agg_log = (
    df_log_allowed
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(count='count', p95_value=lambda x: x.quantile(0.95))
)

agg_log_top20 = agg_log.sort_values('count', ascending=False).head(20)

present_log_components = df_log_allowed['cmdb_id'].unique().tolist()
zero_log_components = sorted([c for c in allowed_components if c not in present_log_components])

# Store aggregated tables in kernel variables (already assigned: agg_metric, agg_trace, agg_log)
# Compact result display (top 20 each and zero-lists)
agg_metric_top20, zero_metric_components, agg_trace_top20, zero_trace_components, agg_log_top20, zero_log_components

```
Out[2]:
```
# Summarize results in plain English using existing aggregated variables
def fmt_row(row, name_field):
    # Format p95_value nicely, handle NaN
    p95 = row.p95_value
    if pd.isna(p95):
        p95_s = "NaN"
    elif abs(p95) >= 1e6 or (isinstance(p95, (int, float)) and float(p95).is_integer()):
        p95_s = f"{int(p95)}"
    else:
        p95_s = f"{p95:.6g}"
    return f"{row.cmdb_id} | {getattr(row, name_field)} | count={int(row['count'])} | p95={p95_s}"

# Prepare top-5 summaries for compactness
top5_metric = agg_metric_top20.head(5)
top5_trace = agg_trace_top20.head(5)
top5_log = agg_log_top20.head(5)

lines = []
lines.append("Summary of aggregated telemetry (compact):")
lines.append("")
lines.append("1) metric.csv — Top 5 (cmdb_id | kpi_name | count | p95_value):")
for _, r in top5_metric.reset_index(drop=True).iterrows():
    lines.append("  - " + fmt_row(r, 'kpi_name'))
lines.append(f"   Allowed components with ZERO metric rows: {zero_metric_components}")
lines.append("")
lines.append("2) trace.csv — Top 5 (cmdb_id | trace_name | count | p95_value):")
for _, r in top5_trace.reset_index(drop=True).iterrows():
    lines.append("  - " + fmt_row(r, 'trace_name'))
lines.append(f"   Allowed components with ZERO trace rows: {zero_trace_components}")
lines.append("")
lines.append("3) log.csv — Top 5 (cmdb_id | log_name | count | p95_value):")
for _, r in top5_log.reset_index(drop=True).iterrows():
    lines.append("  - " + fmt_row(r, 'log_name'))
lines.append(f"   Allowed components with ZERO log rows: {zero_log_components}")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count     p95_value
959    Redis02                             OSLinux-CPU_CPU_CPUWio     31  4.595750e-02
956    Redis02                            OSLinux-CPU_CPU_CPULoad     31  2.050000e-01
957    Redis02                         OSLinux-CPU_CPU_CPUSysTime     31  9.576400e-01
958    Redis02                        OSLinux-CPU_CPU_CPUUserTime     31  3.637000e-01
960    Redis02                        OSLinux-CPU_CPU_CPUidleutil     31  9.905284e+01
94        IG02           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.600000e+01
88        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09
90        IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  8.317770e+08
91        IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.861234e+07
1544  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
1545  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02
1546  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  2.500000e+00
1547  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02
1013   Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00
1014   Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00
1019   Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  3.500000e-01
1020   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  4.000000e+00
1027   Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.321702e+06
1016   Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00
715    Mysql02                Mysql-MySQL_3306_Select Range Check     31  0.000000e+00, [],    cmdb_id                         trace_name  count    p95_value
3     IG01           trace.self.duration_mean     31     6.610702
6     IG01    trace.to_Tomcat01.duration_mean     31     0.367717
5     IG01               trace.self.row_count     31  1058.000000
4     IG01            trace.self.duration_p95     31    60.339950
7     IG01     trace.to_Tomcat01.duration_p95     31     1.998875
13    IG01     trace.to_Tomcat03.duration_p95     31     2.054425
14    IG01        trace.to_Tomcat03.row_count     31   142.500000
8     IG01        trace.to_Tomcat01.row_count     31   133.000000
12    IG01    trace.to_Tomcat03.duration_mean     31     0.363216
30    IG02    trace.to_Tomcat03.duration_mean     31     0.389813
29    IG02        trace.to_Tomcat02.row_count     31   134.500000
28    IG02     trace.to_Tomcat02.duration_p95     31     2.086325
31    IG02     trace.to_Tomcat03.duration_p95     31     2.095150
21    IG02           trace.self.duration_mean     31     7.276158
23    IG02               trace.self.row_count     31  1059.000000
22    IG02            trace.self.duration_p95     31    58.488000
27    IG02    trace.to_Tomcat02.duration_mean     31     0.352452
55    MG01   trace.from_dockerB1.duration_p95     31     0.432450
57    MG01  trace.from_dockerB2.duration_mean     31     0.081691
47    MG01      trace.from_Tomcat04.row_count     31   139.500000, ['Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'apache01', 'apache02'],      cmdb_id          log_name  count  p95_value
9   Tomcat02     log.row_count     31   254.0000
7   Tomcat02   log.error_count     31     0.0000
6   Tomcat01     log.row_count     31   261.0000
4   Tomcat01   log.error_count     31     0.0000
18  apache02   log.error_count     31    14.0000
17  apache01     log.row_count     31   511.0000
19  apache02     log.row_count     31   506.0000
16  apache01   log.error_count     31    14.0000
10  Tomcat03   log.error_count     31     0.0000
15  Tomcat04     log.row_count     31   271.5000
13  Tomcat04   log.error_count     31     0.0000
12  Tomcat03     log.row_count     31   258.5000
14  Tomcat04  log.gc_pause_max     11     0.0850
8   Tomcat02  log.gc_pause_max     10     0.0855
11  Tomcat03  log.gc_pause_max     10     0.1120
5   Tomcat01  log.gc_pause_max      9     0.1040
3    Mysql02     log.row_count      1   358.0000
2    Mysql02   log.error_count      1     0.0000
1    Mysql01     log.row_count      1     3.0000
0    Mysql01   log.error_count      1     0.0000, ['IG01', 'IG02', 'MG01', 'MG02', 'Redis01', 'Redis02'])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables from kernel: df_metric, agg_metric
# Ensure timezone
tz_sh = pytz.timezone('Asia/Shanghai')

# Allowed components
allowed_components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# 1) Filter metric rows by allowed components and time window (UTC+8)
start_dt = tz_sh.localize(pd.to_datetime("2021-03-25 19:00:00"))
end_dt   = tz_sh.localize(pd.to_datetime("2021-03-25 19:30:00"))

# Ensure df_metric has 'dt' timezone-aware column
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz_sh)

df_metric_allowed_window = df_metric[
    (df_metric['cmdb_id'].isin(allowed_components)) &
    (df_metric['dt'] >= start_dt) &
    (df_metric['dt'] <= end_dt)
].copy()

# Ensure numeric value
df_metric_allowed_window['value'] = pd.to_numeric(df_metric_allowed_window['value'], errors='coerce')

# 2) Join with group's global p95_value from agg_metric (assumes agg_metric exists)
# Ensure agg_metric has columns ['cmdb_id','kpi_name','p95_value']
# If agg_metric uses 'value' as column name for p95, adapt accordingly (we used 'p95_value' earlier)
if 'p95_value' not in agg_metric.columns:
    # If previously named differently, attempt to find it
    possible = [c for c in agg_metric.columns if 'p95' in c.lower()]
    if possible:
        agg_metric = agg_metric.rename(columns={possible[0]: 'p95_value'})
    else:
        agg_metric['p95_value'] = np.nan

df_joined = pd.merge(
    df_metric_allowed_window,
    agg_metric[['cmdb_id','kpi_name','p95_value']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# 3) Mark anomaly: value >= p95_value and p95_value not NaN
df_joined['anomaly'] = (~df_joined['p95_value'].isna()) & (df_joined['value'] >= df_joined['p95_value'])

# 4) For each component-kpi group compute required aggregates
grouped = df_joined.groupby(['cmdb_id','kpi_name'], as_index=False)

summary = grouped.agg(
    earliest_anomaly_timestamp = ('dt', lambda s: s[df_joined.loc[s.index, 'anomaly']].min() if s.size>0 else pd.NaT),
    count_anomalies = ('anomaly', 'sum'),
    max_value = ('value', 'max'),
    p95_value = ('p95_value', 'first')  # p95_value is global per group, same for the group
)

# severity_ratio = max_value / p95_value if p95_value>0 else NaN
summary['severity_ratio'] = np.where(summary['p95_value'] > 0, summary['max_value'] / summary['p95_value'], np.nan)

# Normalize earliest_anomaly_timestamp dtype (could be NaT)
# 5) Return top 20 groups sorted by severity_ratio descending
anomaly_summary = summary.sort_values('severity_ratio', ascending=False, na_position='last').reset_index(drop=True)
anomaly_summary_top20 = anomaly_summary[['cmdb_id','kpi_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio']].head(20)

# Also return up to 5 sample anomaly rows across the top groups
top_groups = anomaly_summary_top20[['cmdb_id','kpi_name']].dropna().astype(str)
# Build a boolean mask for rows that belong to any top group
if not top_groups.empty:
    # Merge key for matching
    df_joined['_group_key'] = df_joined['cmdb_id'].astype(str) + "||" + df_joined['kpi_name'].astype(str)
    top_keys = top_groups['cmdb_id'].astype(str) + "||" + top_groups['kpi_name'].astype(str)
    mask = df_joined['_group_key'].isin(top_keys) & (df_joined['anomaly'] == True)
    anomaly_samples_df = df_joined.loc[mask, ['dt','cmdb_id','kpi_name','value']].rename(columns={'dt':'timestamp'})
    # limit to 5 rows total, sort by timestamp ascending for samples
    anomaly_samples = anomaly_samples_df.sort_values('timestamp').head(5).reset_index(drop=True)
else:
    anomaly_samples = df_joined.iloc[0:0][['timestamp','cmdb_id','kpi_name','value']]

# Store results in kernel variables for next steps
# anomaly_summary (full), anomaly_summary_top20, anomaly_samples
anomaly_summary, anomaly_summary_top20, anomaly_samples

```
Out[3]:
```
import pandas as pd

# Use existing kernel variables: anomaly_summary, anomaly_summary_top20, anomaly_samples
# Prepare counts
total_groups = int(anomaly_summary.shape[0]) if 'anomaly_summary' in globals() else 0
groups_with_anomalies = int((anomaly_summary['count_anomalies'] > 0).sum()) if 'anomaly_summary' in globals() else 0
total_anomalies = int(anomaly_summary['count_anomalies'].sum()) if 'anomaly_summary' in globals() else 0

# Top 5 groups from the previously computed top-20 (if available)
top_list = []
if 'anomaly_summary_top20' in globals() and not anomaly_summary_top20.empty:
    top5 = anomaly_summary_top20.head(5)
    for _, r in top5.iterrows():
        ts = r['earliest_anomaly_timestamp']
        ts_s = pd.NaT if pd.isna(ts) else str(ts)
        top_list.append(f"{r['cmdb_id']} | {r['kpi_name']} | earliest={ts_s} | anomalies={int(r['count_anomalies'])} | max={float(r['max_value']):.6g} | p95={float(r['p95_value']):.6g} | severity={float(r['severity_ratio']):.6g}")
else:
    top_list.append("No groups in top list.")

# Up to 5 sample anomaly rows
samples_list = []
if 'anomaly_samples' in globals() and not anomaly_samples.empty:
    for _, s in anomaly_samples.iterrows():
        samples_list.append(f"{s['timestamp']} | {s['cmdb_id']} | {s['kpi_name']} | value={float(s['value']):.6g}")
else:
    samples_list.append("No anomaly samples.")

# Build summary string
summary_lines = [
    "Metric anomaly summary (window 2021-03-25 19:00 to 19:30 UTC+8):",
    f"- Total component-KPI groups examined: {total_groups}",
    f"- Groups with >=1 anomaly in window: {groups_with_anomalies} (total anomaly points across groups: {total_anomalies})",
    "",
    "Top 5 groups by severity_ratio:",
]
summary_lines += [f"  - {line}" for line in top_list]
summary_lines += [
    "",
    "Up to 5 sample anomaly rows (timestamp | cmdb_id | kpi_name | value):",
]
summary_lines += [f"  - {line}" for line in samples_list]

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                         kpi_name earliest_anomaly_timestamp  count_anomalies  max_value  p95_value  severity_ratio
0     apache02   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-25 19:17:00+08:00                1   2.000000   0.100000       20.000000
1     apache01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-25 19:18:00+08:00                1   2.000000   0.200000       10.000000
2     Tomcat03                           OSLinux-CPU_CPU_CPUWio  2021-03-25 19:00:00+08:00                2   2.902287   0.553225        5.246128
3     Tomcat04                           OSLinux-CPU_CPU_CPUWio  2021-03-25 19:01:00+08:00                2   2.839900   0.669515        4.241727
4      Redis02                           OSLinux-CPU_CPU_CPUWio  2021-03-25 19:02:00+08:00                2   0.179563   0.045957        3.907153
...        ...                                              ...                        ...              ...        ...        ...             ...
1704  apache02    OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2021-03-25 19:00:00+08:00               18   0.000000   0.000000             NaN
1705  apache02    OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc  2021-03-25 19:01:00+08:00                9   0.000000   0.000000             NaN
1706  apache02  OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent  2021-03-25 19:01:00+08:00                9   0.000000   0.000000             NaN
1707  apache02                     OSLinux-OSLinux_SWAP_SWAP_Si  2021-03-25 19:03:00+08:00               10   0.000000   0.000000             NaN
1708  apache02                     OSLinux-OSLinux_SWAP_SWAP_So  2021-03-25 19:03:00+08:00               10   0.000000   0.000000             NaN

[1709 rows x 7 columns],      cmdb_id                                           kpi_name earliest_anomaly_timestamp  count_anomalies   max_value   p95_value  severity_ratio
0   apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-25 19:17:00+08:00                1    2.000000    0.100000       20.000000
1   apache01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  2021-03-25 19:18:00+08:00                1    2.000000    0.200000       10.000000
2   Tomcat03                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:00:00+08:00                2    2.902287    0.553225        5.246128
3   Tomcat04                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:01:00+08:00                2    2.839900    0.669515        4.241727
4    Redis02                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:02:00+08:00                2    0.179563    0.045957        3.907153
5       MG01                    OSLinux-CPU_CPU-3_SingleCpuidle  2021-03-25 19:00:00+08:00                1   95.888100   26.152258        3.666532
6    Mysql02                    Mysql-MySQL_3306_ThreadsRunning  2021-03-25 19:00:00+08:00                2   17.000000    5.000000        3.400000
7    Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...  2021-03-25 19:15:00+08:00                1    0.800000    0.240000        3.333333
8   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-25 19:01:00+08:00                1    2.000000    0.600000        3.333333
9       MG02                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:00:00+08:00                2    2.283884    0.703255        3.247590
10  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  2021-03-25 19:16:00+08:00                1  755.600000  233.866690        3.230900
11  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  2021-03-25 19:16:00+08:00                1   12.066700    3.783343        3.189428
12  Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuidle  2021-03-25 19:15:00+08:00                1   29.998300    9.767187        3.071335
13  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  2021-03-25 19:01:00+08:00                1    0.350000    0.128310        2.727769
14  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  2021-03-25 19:11:00+08:00                1    0.183300    0.075010        2.443674
15   Redis02                            OSLinux-CPU_CPU_CPULoad  2021-03-25 19:05:00+08:00                2    0.470000    0.205000        2.292683
16  Tomcat01                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:00:00+08:00                2    2.611800    1.154903        2.261488
17   Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty  2021-03-25 19:00:00+08:00                2  111.000000   52.500000        2.114286
18      IG02                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:30:00+08:00                1    0.889500    0.421980        2.107920
19  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  2021-03-25 19:30:00+08:00                1    8.000000    3.800000        2.105263,                   timestamp   cmdb_id                                         kpi_name      value
0 2021-03-25 19:00:00+08:00      MG01                  OSLinux-CPU_CPU-3_SingleCpuidle  95.888100
1 2021-03-25 19:00:00+08:00      MG02                           OSLinux-CPU_CPU_CPUWio   2.283884
2 2021-03-25 19:00:00+08:00   Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty  87.000000
3 2021-03-25 19:00:00+08:00   Mysql02                  Mysql-MySQL_3306_ThreadsRunning  17.000000
4 2021-03-25 19:00:00+08:00  Tomcat01                           OSLinux-CPU_CPU_CPUWio   2.611800)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel: df_trace, agg_trace, allowed_components (if not present, define)
tz_sh = pytz.timezone('Asia/Shanghai')

if 'allowed_components' not in globals():
    allowed_components = [
        "apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
        "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
    ]

# Ensure df_trace has timezone-aware dt column
if 'dt' not in df_trace.columns:
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz_sh)

# Time window (inclusive) in UTC+8
start_dt = tz_sh.localize(pd.to_datetime("2021-03-25 19:00:00"))
end_dt   = tz_sh.localize(pd.to_datetime("2021-03-25 19:30:00"))

# 1) Filter trace rows by allowed components and time window
df_trace_window = df_trace[
    (df_trace['cmdb_id'].isin(allowed_components)) &
    (df_trace['dt'] >= start_dt) &
    (df_trace['dt'] <= end_dt)
].copy()

# Ensure numeric 'value'
df_trace_window['value'] = pd.to_numeric(df_trace_window['value'], errors='coerce')

# 2) Ensure agg_trace has p95_value column; if not, attempt to rename
if 'p95_value' not in agg_trace.columns:
    possible = [c for c in agg_trace.columns if 'p95' in c.lower()]
    if possible:
        agg_trace = agg_trace.rename(columns={possible[0]: 'p95_value'})
    else:
        agg_trace['p95_value'] = np.nan

# Join each trace row with group's global p95_value
df_trace_joined = pd.merge(
    df_trace_window,
    agg_trace[['cmdb_id','trace_name','p95_value']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# 3) Mark anomaly: value >= p95_value and p95_value not NaN
df_trace_joined['anomaly'] = (~df_trace_joined['p95_value'].isna()) & (df_trace_joined['value'] >= df_trace_joined['p95_value'])

# 4) For each component-trace_name group compute aggregates
grouped = df_trace_joined.groupby(['cmdb_id','trace_name'], as_index=False)

trace_agg = grouped.agg(
    count_anomalies = ('anomaly', 'sum'),
    max_value = ('value', 'max'),
    p95_value = ('p95_value', 'first')
)

# earliest anomaly timestamp computed from anomalous rows only
earliest = (
    df_trace_joined[df_trace_joined['anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(earliest_anomaly_timestamp = ('dt', 'min'))
)

# merge earliest into trace_agg
trace_anomaly_summary = pd.merge(trace_agg, earliest, on=['cmdb_id','trace_name'], how='left')

# severity_ratio = max_value / p95_value if p95_value > 0 else NaN
trace_anomaly_summary['severity_ratio'] = np.where(
    trace_anomaly_summary['p95_value'] > 0,
    trace_anomaly_summary['max_value'] / trace_anomaly_summary['p95_value'],
    np.nan
)

# 5) Top 20 groups sorted by severity_ratio descending
trace_anomaly_top20 = (
    trace_anomaly_summary
    .sort_values('severity_ratio', ascending=False, na_position='last')
    [['cmdb_id','trace_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# Sample up to 5 anomalous trace rows across the top groups (by timestamp asc)
top_keys = trace_anomaly_top20[['cmdb_id','trace_name']].astype(str)
if not top_keys.empty:
    df_trace_joined['_group_key'] = df_trace_joined['cmdb_id'].astype(str) + "||" + df_trace_joined['trace_name'].astype(str)
    top_keys_series = top_keys['cmdb_id'].astype(str) + "||" + top_keys['trace_name'].astype(str)
    mask = df_trace_joined['_group_key'].isin(top_keys_series) & (df_trace_joined['anomaly'] == True)
    trace_anomaly_samples = (
        df_trace_joined.loc[mask, ['dt','cmdb_id','trace_name','value']]
        .rename(columns={'dt':'timestamp'})
        .sort_values('timestamp')
        .head(5)
        .reset_index(drop=True)
    )
else:
    trace_anomaly_samples = df_trace_joined.loc[0:0, ['dt','cmdb_id','trace_name','value']].rename(columns={'dt':'timestamp'})

# Store variables in kernel for next steps:
# trace_anomaly_summary (full), trace_anomaly_top20 (top 20), trace_anomaly_samples (up to 5 rows)

trace_anomaly_summary, trace_anomaly_top20, trace_anomaly_samples

```
Out[4]:
```
import pandas as pd

# Use existing kernel variables: trace_anomaly_summary, trace_anomaly_top20, trace_anomaly_samples
total_groups = int(trace_anomaly_summary.shape[0]) if 'trace_anomaly_summary' in globals() else 0
groups_with_anomalies = int((trace_anomaly_summary['count_anomalies'] > 0).sum()) if 'trace_anomaly_summary' in globals() else 0
total_anomaly_points = int(trace_anomaly_summary['count_anomalies'].sum()) if 'trace_anomaly_summary' in globals() else 0

# Top 5 from previously computed top-20 (if available)
top_lines = []
if 'trace_anomaly_top20' in globals() and not trace_anomaly_top20.empty:
    for _, r in trace_anomaly_top20.head(5).iterrows():
        ts = r['earliest_anomaly_timestamp']
        ts_s = "None" if pd.isna(ts) else str(ts)
        top_lines.append(
            f"{r['cmdb_id']} | {r['trace_name']} | earliest={ts_s} | anomalies={int(r['count_anomalies'])} | "
            f"max={float(r['max_value']):.6g} | p95={float(r['p95_value']):.6g} | severity={float(r['severity_ratio']):.6g}"
        )
else:
    top_lines.append("No top groups available.")

# Up to 5 sample anomalous trace rows
sample_lines = []
if 'trace_anomaly_samples' in globals() and not trace_anomaly_samples.empty:
    for _, s in trace_anomaly_samples.iterrows():
        sample_lines.append(f"{s['timestamp']} | {s['cmdb_id']} | {s['trace_name']} | value={float(s['value']):.6g}")
else:
    sample_lines.append("No anomaly samples.")

summary_lines = [
    "Trace anomaly summary (window 2021-03-25 19:00 to 19:30 UTC+8):",
    f"- Total component-trace groups examined: {total_groups}",
    f"- Groups with >=1 anomaly in window: {groups_with_anomalies} (total anomalous points across groups: {total_anomaly_points})",
    "",
    "Top 5 groups by severity_ratio:",
]
summary_lines += [f"  - {l}" for l in top_lines]
summary_lines += [
    "",
    "Up to 5 sample anomalous trace rows (timestamp | cmdb_id | trace_name | value):",
]
summary_lines += [f"  - {l}" for l in sample_lines]

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                     trace_name  count_anomalies   max_value   p95_value earliest_anomaly_timestamp  severity_ratio
0        IG01  trace.from_root.duration_mean                1    1.028667    0.977984  2021-03-25 19:00:00+08:00        1.051824
1        IG01   trace.from_root.duration_p95                1    1.941500    1.845175  2021-03-25 19:00:00+08:00        1.052204
2        IG01      trace.from_root.row_count                1    3.000000    2.900000  2021-03-25 19:00:00+08:00        1.034483
3        IG01       trace.self.duration_mean                2    7.312182    6.610702  2021-03-25 19:15:00+08:00        1.106113
4        IG01        trace.self.duration_p95                2   61.009000   60.339950  2021-03-25 19:15:00+08:00        1.011088
..        ...                            ...              ...         ...         ...                        ...             ...
187  Tomcat04     trace.to_MG01.duration_p95                2    2.611800    1.943550  2021-03-25 19:00:00+08:00        1.343830
188  Tomcat04        trace.to_MG01.row_count                2  148.000000  139.500000  2021-03-25 19:20:00+08:00        1.060932
189  Tomcat04    trace.to_MG02.duration_mean                2    0.291490    0.280734  2021-03-25 19:06:00+08:00        1.038312
190  Tomcat04     trace.to_MG02.duration_p95                2    2.089400    2.014000  2021-03-25 19:00:00+08:00        1.037438
191  Tomcat04        trace.to_MG02.row_count                2  138.000000  135.500000  2021-03-25 19:21:00+08:00        1.018450

[192 rows x 7 columns],      cmdb_id                         trace_name earliest_anomaly_timestamp  count_anomalies  max_value  p95_value  severity_ratio
0       MG01  trace.from_dockerA2.duration_mean  2021-03-25 19:00:00+08:00                2   2.193000   0.136725       16.039513
1   Tomcat03            trace.self.duration_p95  2021-03-25 19:07:00+08:00                2   0.906100   0.063425       14.286165
2   Tomcat03      trace.from_IG01.duration_mean  2021-03-25 19:07:00+08:00                2   1.065000   0.363216        2.932144
3       IG01    trace.to_Tomcat03.duration_mean  2021-03-25 19:07:00+08:00                2   1.065000   0.363216        2.932144
4       MG01   trace.from_dockerA2.duration_p95  2021-03-25 19:00:00+08:00                2   2.193000   0.973257        2.253258
5   Tomcat04            trace.self.duration_p95  2021-03-25 19:29:00+08:00                2   0.161000   0.074450        2.162525
6       MG02  trace.from_Tomcat01.duration_mean  2021-03-25 19:28:00+08:00                2   0.605500   0.282754        2.141437
7   Tomcat01        trace.to_MG02.duration_mean  2021-03-25 19:28:00+08:00                2   0.605500   0.282754        2.141437
8       MG01     trace.to_dockerB2.duration_p95  2021-03-25 19:13:00+08:00                2   6.125200   3.049675        2.008476
9       MG01   trace.from_dockerB1.duration_p95  2021-03-25 19:00:00+08:00                2   0.828000   0.432450        1.914672
10      MG02  trace.from_dockerA1.duration_mean  2021-03-25 19:00:00+08:00                2   0.181889   0.102871        1.768134
11  Tomcat02         trace.to_MG01.duration_p95  2021-03-25 19:17:00+08:00                2   2.081300   1.209515        1.720772
12      MG01   trace.from_Tomcat02.duration_p95  2021-03-25 19:17:00+08:00                2   2.081300   1.209515        1.720772
13      MG02  trace.from_dockerB2.duration_mean  2021-03-25 19:00:00+08:00                2   0.190642   0.113128        1.685183
14      MG01   trace.from_dockerB2.duration_p95  2021-03-25 19:00:00+08:00                2   0.589050   0.360475        1.634094
15  Tomcat03           trace.self.duration_mean  2021-03-25 19:07:00+08:00                2   0.092565   0.058108        1.592982
16      MG01   trace.from_dockerA1.duration_p95  2021-03-25 19:00:00+08:00                2   1.017000   0.640870        1.586905
17      MG01  trace.from_dockerB1.duration_mean  2021-03-25 19:00:00+08:00                2   0.114398   0.073007        1.566946
18      MG02   trace.from_dockerA1.duration_p95  2021-03-25 19:00:00+08:00                2   1.013000   0.651330        1.555279
19      MG01           trace.self.duration_mean  2021-03-25 19:00:00+08:00                2   0.248562   0.161594        1.538193,                   timestamp cmdb_id                         trace_name     value
0 2021-03-25 19:00:00+08:00    MG01   trace.from_dockerA1.duration_p95  0.745900
1 2021-03-25 19:00:00+08:00    MG02  trace.from_dockerB2.duration_mean  0.114095
2 2021-03-25 19:00:00+08:00    MG02  trace.from_dockerA1.duration_mean  0.181889
3 2021-03-25 19:00:00+08:00    MG01           trace.self.duration_mean  0.172493
4 2021-03-25 19:00:00+08:00    MG01   trace.from_dockerB2.duration_p95  0.492000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing DataFrames/variables from kernel: df_error_logs, df_log, trace_anomaly_summary, trace_anomaly_samples
tz_sh = pytz.timezone('Asia/Shanghai')

# Ensure dt columns exist and are timezone-aware UTC+8
for df in (df_error_logs, df_log):
    if 'dt' not in df.columns and 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz_sh)

# Ensure trace_anomaly_summary and trace_anomaly_samples exist and have proper timestamp columns
if 'trace_anomaly_summary' in globals():
    # earliest_anomaly_timestamp may already be timezone-aware; if not, try to convert
    if 'earliest_anomaly_timestamp' in trace_anomaly_summary.columns:
        # convert naive to tz-aware if needed (only if not tz-aware)
        if pd.api.types.is_datetime64_any_dtype(trace_anomaly_summary['earliest_anomaly_timestamp']):
            # make sure tz aware
            if trace_anomaly_summary['earliest_anomaly_timestamp'].dt.tz is None:
                trace_anomaly_summary['earliest_anomaly_timestamp'] = trace_anomaly_summary['earliest_anomaly_timestamp'].dt.tz_localize(tz_sh)
if 'trace_anomaly_samples' in globals():
    if 'timestamp' in trace_anomaly_samples.columns:
        if pd.api.types.is_datetime64_any_dtype(trace_anomaly_samples['timestamp']):
            if trace_anomaly_samples['timestamp'].dt.tz is None:
                trace_anomaly_samples['timestamp'] = trace_anomaly_samples['timestamp'].dt.tz_localize(tz_sh)

# Define window (inclusive)
start_dt = tz_sh.localize(pd.to_datetime("2021-03-25 19:14:00"))
end_dt   = tz_sh.localize(pd.to_datetime("2021-03-25 19:20:00"))

# 1) error_logs.csv: filter rows in window for apache01 and apache02
apache_nodes = ['apache01','apache02']
err_window = df_error_logs[
    (df_error_logs['cmdb_id'].isin(apache_nodes)) &
    (df_error_logs['dt'] >= start_dt) &
    (df_error_logs['dt'] <= end_dt)
].copy()

# Per-minute counts grouped by minute timestamp and cmdb_id (only minutes with >0 entries)
# Floor dt to minute
err_window['minute'] = err_window['dt'].dt.floor('T')
error_per_minute = (
    err_window.groupby(['minute','cmdb_id'], as_index=False)
    .size()
    .rename(columns={'size':'count'})
    .sort_values(['minute','cmdb_id'], ascending=[True, True])
)
# Keep only minutes with >0 (groupby.size already >0)
# Total count per cmdb_id in interval
error_totals = (
    err_window.groupby('cmdb_id', as_index=False)
    .size()
    .rename(columns={'size':'total_count'})
    .sort_values('total_count', ascending=False)
)

# Top 10 distinct messages across both apache nodes with counts
# If message column exists
if 'message' in err_window.columns and not err_window['message'].isna().all():
    error_top_messages = (
        err_window['message']
        .value_counts()
        .rename_axis('message')
        .reset_index(name='count')
        .head(10)
    )
else:
    # Empty DataFrame with the expected columns
    error_top_messages = pd.DataFrame(columns=['message','count'])

# 2) log.csv: filter rows in same interval for apache nodes and return rows where log.error_count>0
# Ensure dt exists
if 'dt' not in df_log.columns and 'timestamp' in df_log.columns:
    df_log['dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz_sh)

log_window = df_log[
    (df_log['cmdb_id'].isin(apache_nodes)) &
    (df_log['dt'] >= start_dt) &
    (df_log['dt'] <= end_dt)
].copy()

# Ensure numeric 'value'
log_window['value'] = pd.to_numeric(log_window['value'], errors='coerce')

# Filter rows where log_name == 'log.error_count' and value > 0
log_errors_rows = log_window[
    (log_window['log_name'] == 'log.error_count') &
    (log_window['value'] > 0)
].loc[:, ['dt','cmdb_id','log_name','value']].rename(columns={'dt':'timestamp'}).sort_values('timestamp').head(20).reset_index(drop=True)

# 3) trace anomaly summary: filter groups whose earliest_anomaly_timestamp falls within window
if 'trace_anomaly_summary' in globals():
    # ensure earliest_anomaly_timestamp is tz-aware datetime (we attempted above)
    taf = trace_anomaly_summary.copy()
    # Filter rows where earliest_anomaly_timestamp is within window
    trace_anom_window = taf[
        (taf['earliest_anomaly_timestamp'].notna()) &
        (taf['earliest_anomaly_timestamp'] >= start_dt) &
        (taf['earliest_anomaly_timestamp'] <= end_dt)
    ].loc[:, ['cmdb_id','trace_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio']]

    trace_anom_window = trace_anom_window.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)
else:
    trace_anom_window = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio'])

# Also return up to 5 sample anomalous trace rows from trace_anomaly_samples within same window
if 'trace_anomaly_samples' in globals() and not trace_anomaly_samples.empty:
    tas = trace_anomaly_samples.copy()
    # ensure timestamp tz-aware handled above
    trace_anom_samples_window = tas[
        (tas['timestamp'] >= start_dt) &
        (tas['timestamp'] <= end_dt)
    ].loc[:, ['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').head(5).reset_index(drop=True)
else:
    trace_anom_samples_window = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])

# Store results in kernel variables for next steps and display compact outputs
error_per_minute, error_totals, error_top_messages, log_errors_rows, trace_anom_window, trace_anom_samples_window

```
Out[5]:
```
summary = (
    "Mid-window findings (2021-03-25 19:14–19:20 UTC+8) for apache01 & apache02:\n\n"
    "1) error_logs per-minute counts (minutes with >0 entries):\n"
    "   - 2021-03-25 19:15: apache01=30, apache02=14\n"
    "   - 2021-03-25 19:16: apache01=11, apache02=13\n"
    "   - 2021-03-25 19:17: apache01=17, apache02=16\n"
    "   - 2021-03-25 19:18: apache01=10, apache02=14\n"
    "   - 2021-03-25 19:19: apache01=2,  apache02=2\n\n"
    "   Totals in interval: apache01 = 70, apache02 = 59\n\n"
    "   Top distinct messages (top 10 across both nodes): mostly unique HTTP POST request entries (each count = 1), e.g. messages containing\n"
    "   'POST /UOCP/person/ServiceTest10.json', 'POST /UOCP/person/ServiceTest1.json', 'POST /UOCP/person/ServiceTest9.json', etc.\n\n"
    "2) log.csv (same interval) — rows with log.error_count > 0 (up to 20 shown):\n"
    "   - Matches the per-minute error counts above: entries at 19:15,19:16,19:17,19:18,19:19 for both apache01 and apache02\n"
    "   - Example rows: 19:15 apache01 log.error_count=30; 19:15 apache02 log.error_count=14; 19:16 apache01=11; 19:16 apache02=13; etc.\n\n"
    "3) trace anomaly groups with earliest_anomaly_timestamp inside window (up to 20 rows, sorted by severity_ratio):\n"
    "   - Top entries include:\n"
    "     * MG01 | trace.from_Tomcat02.duration_p95 — earliest 19:17, anomalies=2, max=2.0813, p95=1.2095, severity≈1.7208\n"
    "     * Tomcat02 | trace.to_MG01.duration_p95 — earliest 19:17, anomalies=2, max=2.0813, p95=1.2095, severity≈1.7208\n"
    "     * Tomcat01 | trace.to_MG01.duration_p95 — earliest 19:17, anomalies=2, max=2.0854, p95=1.6396, severity≈1.2719\n"
    "     * MG01 | trace.from_Tomcat01.duration_p95 — earliest 19:17, anomalies=2, max=2.0854, p95=1.6396, severity≈1.2719\n"
    "     * IG01 | trace.self.duration_mean — earliest 19:15, anomalies=2, max≈7.312, p95≈6.611, severity≈1.106\n"
    "   - Full returned table contains up to 20 groups (severity ratios range from ≈1.72 down to ≈1.05 in the selection).\n\n"
    "   Sample anomalous trace rows within the same window: none returned (no sample rows matched the top-group filter in this exact interval).\n\n"
    "Summary interpretation:\n"
    "- Both apache01 and apache02 show a clear spike in error log counts around 19:15–19:18, with apache01 slightly higher overall (70 vs 59).\n"
    "- log.error_count entries confirm the same per-minute spikes.\n"
    "- Several trace groups involving MG and Tomcat components show anomalous behavior with elevated durations around 19:17 (severity ratios up to ~1.72), and IG01 shows trace duration elevation at ~19:15.\n"
    "- No individual anomalous trace sample rows were returned for the top groups within the exact sample selection window.\n\n"
    "If you want, I can (a) show the full top-20 trace-anomaly rows table, (b) display the full top messages list, or (c) extract raw error log lines for a given minute."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     minute   cmdb_id  count
0 2021-03-25 19:15:00+08:00  apache01     30
1 2021-03-25 19:15:00+08:00  apache02     14
2 2021-03-25 19:16:00+08:00  apache01     11
3 2021-03-25 19:16:00+08:00  apache02     13
4 2021-03-25 19:17:00+08:00  apache01     17
5 2021-03-25 19:17:00+08:00  apache02     16
6 2021-03-25 19:18:00+08:00  apache01     10
7 2021-03-25 19:18:00+08:00  apache02     14
8 2021-03-25 19:19:00+08:00  apache01      2
9 2021-03-25 19:19:00+08:00  apache02      2,     cmdb_id  total_count
0  apache01           70
1  apache02           59,                                              message  count
0  IPAddress "POST /UOCP/person/ServiceTest10.jso...      1
1  IPAddress "POST /UOCP/person/ServiceTest1.json...      1
2  IPAddress "POST /UOCP/person/ServiceTest9.json...      1
3  IPAddress "POST /UOCP/base/ServiceTest7.json H...      1
4  IPAddress "POST /UOCP/person/ServiceTest1.json...      1
5  IPAddress "POST /UOCP/base/ServiceTest8.json H...      1
6  IPAddress "POST /UOCP/base/ServiceTest2.json H...      1
7  IPAddress "POST /UOCP/person/ServiceTest1.json...      1
8  IPAddress "POST /UOCP/person/ServiceTest5.json...      1
9  IPAddress "POST /UOCP/person/ServiceTest10.jso...      1,                   timestamp   cmdb_id         log_name  value
0 2021-03-25 19:15:00+08:00  apache01  log.error_count   30.0
1 2021-03-25 19:15:00+08:00  apache02  log.error_count   14.0
2 2021-03-25 19:16:00+08:00  apache01  log.error_count   11.0
3 2021-03-25 19:16:00+08:00  apache02  log.error_count   13.0
4 2021-03-25 19:17:00+08:00  apache01  log.error_count   17.0
5 2021-03-25 19:17:00+08:00  apache02  log.error_count   16.0
6 2021-03-25 19:18:00+08:00  apache01  log.error_count   10.0
7 2021-03-25 19:18:00+08:00  apache02  log.error_count   14.0
8 2021-03-25 19:19:00+08:00  apache01  log.error_count    2.0
9 2021-03-25 19:19:00+08:00  apache02  log.error_count    2.0,      cmdb_id                         trace_name earliest_anomaly_timestamp  count_anomalies   max_value   p95_value  severity_ratio
0       MG01   trace.from_Tomcat02.duration_p95  2021-03-25 19:17:00+08:00                2    2.081300    1.209515        1.720772
1   Tomcat02         trace.to_MG01.duration_p95  2021-03-25 19:17:00+08:00                2    2.081300    1.209515        1.720772
2   Tomcat01         trace.to_MG01.duration_p95  2021-03-25 19:17:00+08:00                2    2.085400    1.639550        1.271934
3       MG01   trace.from_Tomcat01.duration_p95  2021-03-25 19:17:00+08:00                2    2.085400    1.639550        1.271934
4       MG01  trace.from_Tomcat02.duration_mean  2021-03-25 19:17:00+08:00                2    0.305441    0.255591        1.195036
5   Tomcat02        trace.to_MG01.duration_mean  2021-03-25 19:17:00+08:00                2    0.305441    0.255591        1.195036
6   Tomcat01        trace.to_MG01.duration_mean  2021-03-25 19:17:00+08:00                2    0.310303    0.266153        1.165880
7       MG01  trace.from_Tomcat01.duration_mean  2021-03-25 19:17:00+08:00                2    0.310303    0.266153        1.165880
8       IG01           trace.self.duration_mean  2021-03-25 19:15:00+08:00                2    7.312182    6.610702        1.106113
9       MG02    trace.to_dockerA2.duration_mean  2021-03-25 19:20:00+08:00                2    0.109200    0.099279        1.099925
10      MG01        trace.to_dockerB1.row_count  2021-03-25 19:20:00+08:00                2  206.000000  191.000000        1.078534
11      IG02           trace.self.duration_mean  2021-03-25 19:15:00+08:00                2    7.813650    7.276158        1.073870
12      IG02            trace.self.duration_p95  2021-03-25 19:15:00+08:00                2   62.058000   58.488000        1.061038
13  Tomcat04            trace.to_MG01.row_count  2021-03-25 19:20:00+08:00                2  148.000000  139.500000        1.060932
14      MG01      trace.from_Tomcat04.row_count  2021-03-25 19:20:00+08:00                2  148.000000  139.500000        1.060932
15      MG01        trace.to_dockerB2.row_count  2021-03-25 19:20:00+08:00                2   68.000000   64.500000        1.054264
16      IG01            trace.self.duration_p95  2021-03-25 19:15:00+08:00                2   61.009000   60.339950        1.011088, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables: anomaly_summary, anomaly_samples, trace_anomaly_summary, trace_anomaly_samples
tz_sh = pytz.timezone('Asia/Shanghai')

# Define early-window boundaries (inclusive) in UTC+8
start_dt = tz_sh.localize(pd.to_datetime("2021-03-25 19:00:00"))
end_dt   = tz_sh.localize(pd.to_datetime("2021-03-25 19:05:00"))

# Helper to ensure a datetime column is timezone-aware (Asia/Shanghai)
def ensure_tz(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        if series.dt.tz is None:
            return series.dt.tz_localize(tz_sh)
        else:
            return series.dt.tz_convert(tz_sh)
    else:
        return series

# 1) Metric groups whose earliest_anomaly_timestamp in 19:00-19:05
# Ensure column exists
if 'anomaly_summary' in globals() and 'earliest_anomaly_timestamp' in anomaly_summary.columns:
    mdf = anomaly_summary.copy()
    mdf['earliest_anomaly_timestamp'] = ensure_tz(mdf['earliest_anomaly_timestamp'])
    early_metric_groups = mdf[
        (mdf['earliest_anomaly_timestamp'].notna()) &
        (mdf['earliest_anomaly_timestamp'] >= start_dt) &
        (mdf['earliest_anomaly_timestamp'] <= end_dt)
    ].loc[:, ['cmdb_id','kpi_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio']]
    early_metric_top20 = early_metric_groups.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)
else:
    early_metric_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio'])

# 2) Up to 5 sample metric anomaly rows from anomaly_samples matching the top metric groups and within window
if 'anomaly_samples' in globals() and not anomaly_samples.empty and not early_metric_top20.empty:
    samp = anomaly_samples.copy()
    # Ensure timestamp tz-aware
    samp['timestamp'] = ensure_tz(samp['timestamp'])
    # Build group key sets for top metric groups
    top_metric_keys = set((early_metric_top20['cmdb_id'].astype(str) + "||" + early_metric_top20['kpi_name'].astype(str)).tolist())
    samp['_key'] = samp['cmdb_id'].astype(str) + "||" + samp['kpi_name'].astype(str)
    early_metric_samples = samp[
        (samp['_key'].isin(top_metric_keys)) &
        (samp['timestamp'] >= start_dt) &
        (samp['timestamp'] <= end_dt)
    ].loc[:, ['timestamp','cmdb_id','kpi_name','value']].sort_values('timestamp').head(5).reset_index(drop=True)
else:
    # empty df with expected cols
    early_metric_samples = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value'])

# 3) Trace groups whose earliest_anomaly_timestamp in 19:00-19:05
if 'trace_anomaly_summary' in globals() and 'earliest_anomaly_timestamp' in trace_anomaly_summary.columns:
    tdf = trace_anomaly_summary.copy()
    tdf['earliest_anomaly_timestamp'] = ensure_tz(tdf['earliest_anomaly_timestamp'])
    early_trace_groups = tdf[
        (tdf['earliest_anomaly_timestamp'].notna()) &
        (tdf['earliest_anomaly_timestamp'] >= start_dt) &
        (tdf['earliest_anomaly_timestamp'] <= end_dt)
    ].loc[:, ['cmdb_id','trace_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio']]
    early_trace_top20 = early_trace_groups.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)
else:
    early_trace_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anomaly_timestamp','count_anomalies','max_value','p95_value','severity_ratio'])

# 4) Up to 5 sample trace anomaly rows from trace_anomaly_samples matching the top trace groups and within window
if 'trace_anomaly_samples' in globals() and not trace_anomaly_samples.empty and not early_trace_top20.empty:
    tsamp = trace_anomaly_samples.copy()
    tsamp['timestamp'] = ensure_tz(tsamp['timestamp'])
    top_trace_keys = set((early_trace_top20['cmdb_id'].astype(str) + "||" + early_trace_top20['trace_name'].astype(str)).tolist())
    tsamp['_key'] = tsamp['cmdb_id'].astype(str) + "||" + tsamp['trace_name'].astype(str)
    early_trace_samples = tsamp[
        (tsamp['_key'].isin(top_trace_keys)) &
        (tsamp['timestamp'] >= start_dt) &
        (tsamp['timestamp'] <= end_dt)
    ].loc[:, ['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').head(5).reset_index(drop=True)
else:
    early_trace_samples = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])

# Store results in kernel variables for next steps (names: early_metric_top20, early_metric_samples, early_trace_top20, early_trace_samples)
early_metric_top20, early_metric_samples, early_trace_top20, early_trace_samples

```
Out[6]:
```
summary = (
    "Early-window (2021-03-25 19:00–19:05 UTC+8) findings (compact):\n\n"
    "1) Metric anomaly groups (top 20 by severity_ratio): 20 groups returned.\n"
    "   Top 5 by severity_ratio:\n"
    "     - Tomcat03 | OSLinux-CPU_CPU_CPUWio | earliest=19:00 | anomalies=2 | max=2.9023 | p95=0.5532 | severity≈5.246\n"
    "     - Tomcat04 | OSLinux-CPU_CPU_CPUWio | earliest=19:01 | anomalies=2 | max=2.8399 | p95=0.6695 | severity≈4.242\n"
    "     - Redis02  | OSLinux-CPU_CPU_CPUWio | earliest=19:02 | anomalies=2 | max=0.1796 | p95=0.0460 | severity≈3.907\n"
    "     - MG01     | OSLinux-CPU_CPU-3_SingleCpuidle | earliest=19:00 | anomalies=1 | max=95.8881 | p95=26.1523 | severity≈3.667\n"
    "     - Mysql02  | Mysql-MySQL_3306_ThreadsRunning | earliest=19:00 | anomalies=2 | max=17.0 | p95=5.0 | severity≈3.400\n\n"
    "   Interpretation: early anomalies concentrate on Tomcat hosts' CPU WIO metrics (high severity ratios) and a large single MG01 CPU idle spike. MySQL threads also elevated.\n\n"
    "2) Sample metric anomaly rows (up to 5 within 19:00–19:05):\n"
    "   - 2021-03-25 19:00 | MG01 | OSLinux-CPU_CPU-3_SingleCpuidle | value=95.8881\n"
    "   - 2021-03-25 19:00 | MG02 | OSLinux-CPU_CPU_CPUWio           | value=2.283884\n"
    "   - 2021-03-25 19:00 | Mysql01 | Mysql-MySQL_3306_Innodb buffer pool pages dirty | value=87.0\n"
    "   - 2021-03-25 19:00 | Mysql02 | Mysql-MySQL_3306_ThreadsRunning | value=17.0\n"
    "   - 2021-03-25 19:00 | Tomcat01 | OSLinux-CPU_CPU_CPUWio        | value=2.6118\n\n"
    "3) Trace anomaly groups (top 20 by severity_ratio) whose earliest anomaly falls in 19:00–19:05: 20 groups returned.\n"
    "   Top 5 by severity_ratio:\n"
    "     - MG01 | trace.from_dockerA2.duration_mean  | earliest=19:00 | anomalies=2 | max=2.1930 | p95=0.1367 | severity≈16.04\n"
    "     - MG01 | trace.from_dockerA2.duration_p95   | earliest=19:00 | anomalies=2 | max=2.1930 | p95=0.9733 | severity≈2.253\n"
    "     - MG01 | trace.from_dockerB1.duration_p95   | earliest=19:00 | anomalies=2 | max=0.8280 | p95=0.4325 | severity≈1.915\n"
    "     - MG02 | trace.from_dockerA1.duration_mean  | earliest=19:00 | anomalies=2 | max=0.1819 | p95=0.1029 | severity≈1.768\n"
    "     - MG02 | trace.from_dockerB2.duration_mean  | earliest=19:00 | anomalies=2 | max=0.1906 | p95=0.1131 | severity≈1.685\n\n"
    "   Interpretation: many early trace anomalies center on MG nodes (interactions with docker components), with one very large relative increase (MG01 trace.from_dockerA2.duration_mean severity ≈16x).\n\n"
    "4) Sample trace anomaly rows (up to 5 within 19:00–19:05):\n"
    "   - 2021-03-25 19:00 | MG01 | trace.from_dockerA1.duration_p95 | value=0.7459\n"
    "   - 2021-03-25 19:00 | MG02 | trace.from_dockerB2.duration_mean | value=0.114095\n"
    "   - 2021-03-25 19:00 | MG02 | trace.from_dockerA1.duration_mean | value=0.181889\n"
    "   - 2021-03-25 19:00 | MG01 | trace.self.duration_mean         | value=0.172493\n"
    "   - 2021-03-25 19:00 | MG01 | trace.from_dockerB2.duration_p95 | value=0.492000\n\n"
    "Overall conclusion (early window):\n"
    "- Early anomalies (19:00–19:05) show strong signals on Tomcat CPU WIO metrics and MG trace durations. The metric severity is highest on Tomcat03/Tomcat04 CPU WIO.\n"
    "- MG nodes exhibit large relative trace-duration increases (notably MG01), suggesting downstream/MG-related latency around 19:00.\n"
    "- These early issues likely contributed to the later apache error spikes observed mid-window. Further drill-down on Tomcat I/O and MG trace paths is recommended.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name earliest_anomaly_timestamp  count_anomalies   max_value  p95_value  severity_ratio
0   Tomcat03                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:00:00+08:00                2    2.902287   0.553225        5.246128
1   Tomcat04                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:01:00+08:00                2    2.839900   0.669515        4.241727
2    Redis02                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:02:00+08:00                2    0.179563   0.045957        3.907153
3       MG01                    OSLinux-CPU_CPU-3_SingleCpuidle  2021-03-25 19:00:00+08:00                1   95.888100  26.152258        3.666532
4    Mysql02                    Mysql-MySQL_3306_ThreadsRunning  2021-03-25 19:00:00+08:00                2   17.000000   5.000000        3.400000
5   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  2021-03-25 19:01:00+08:00                1    2.000000   0.600000        3.333333
6       MG02                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:00:00+08:00                2    2.283884   0.703255        3.247590
7   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  2021-03-25 19:01:00+08:00                1    0.350000   0.128310        2.727769
8    Redis02                            OSLinux-CPU_CPU_CPULoad  2021-03-25 19:05:00+08:00                2    0.470000   0.205000        2.292683
9   Tomcat01                             OSLinux-CPU_CPU_CPUWio  2021-03-25 19:00:00+08:00                2    2.611800   1.154903        2.261488
10   Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages dirty  2021-03-25 19:00:00+08:00                2  111.000000  52.500000        2.114286
11      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  2021-03-25 19:05:00+08:00                1   21.600000  10.680000        2.022472
12   Mysql01                    Mysql-MySQL_3306_ThreadsRunning  2021-03-25 19:00:00+08:00               31    2.000000   1.000000        2.000000
13  Tomcat02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2021-03-25 19:00:00+08:00               31    6.000000   3.000000        2.000000
14      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  2021-03-25 19:02:00+08:00                1    1.000000   0.500000        2.000000
15   Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime  2021-03-25 19:00:00+08:00                2    1.000000   0.500000        2.000000
16  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  2021-03-25 19:01:00+08:00                1    1.000000   0.500000        2.000000
17  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...  2021-03-25 19:00:00+08:00                1    3.400000   1.750000        1.942857
18  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  2021-03-25 19:03:00+08:00                1    1.000000   0.550000        1.818182
19   Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  2021-03-25 19:02:00+08:00                1    1.000000   0.550000        1.818182,                   timestamp   cmdb_id                                         kpi_name      value
0 2021-03-25 19:00:00+08:00      MG01                  OSLinux-CPU_CPU-3_SingleCpuidle  95.888100
1 2021-03-25 19:00:00+08:00      MG02                           OSLinux-CPU_CPU_CPUWio   2.283884
2 2021-03-25 19:00:00+08:00   Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages dirty  87.000000
3 2021-03-25 19:00:00+08:00   Mysql02                  Mysql-MySQL_3306_ThreadsRunning  17.000000
4 2021-03-25 19:00:00+08:00  Tomcat01                           OSLinux-CPU_CPU_CPUWio   2.611800,      cmdb_id                         trace_name earliest_anomaly_timestamp  count_anomalies  max_value  p95_value  severity_ratio
0       MG01  trace.from_dockerA2.duration_mean  2021-03-25 19:00:00+08:00                2   2.193000   0.136725       16.039513
1       MG01   trace.from_dockerA2.duration_p95  2021-03-25 19:00:00+08:00                2   2.193000   0.973257        2.253258
2       MG01   trace.from_dockerB1.duration_p95  2021-03-25 19:00:00+08:00                2   0.828000   0.432450        1.914672
3       MG02  trace.from_dockerA1.duration_mean  2021-03-25 19:00:00+08:00                2   0.181889   0.102871        1.768134
4       MG02  trace.from_dockerB2.duration_mean  2021-03-25 19:00:00+08:00                2   0.190642   0.113128        1.685183
5       MG01   trace.from_dockerB2.duration_p95  2021-03-25 19:00:00+08:00                2   0.589050   0.360475        1.634094
6       MG01   trace.from_dockerA1.duration_p95  2021-03-25 19:00:00+08:00                2   1.017000   0.640870        1.586905
7       MG01  trace.from_dockerB1.duration_mean  2021-03-25 19:00:00+08:00                2   0.114398   0.073007        1.566946
8       MG02   trace.from_dockerA1.duration_p95  2021-03-25 19:00:00+08:00                2   1.013000   0.651330        1.555279
9       MG01           trace.self.duration_mean  2021-03-25 19:00:00+08:00                2   0.248562   0.161594        1.538193
10  Tomcat04        trace.to_MG01.duration_mean  2021-03-25 19:00:00+08:00                2   0.470165   0.306090        1.536033
11      MG01  trace.from_Tomcat04.duration_mean  2021-03-25 19:00:00+08:00                2   0.470165   0.306090        1.536033
12      IG01     trace.to_Tomcat02.duration_p95  2021-03-25 19:00:00+08:00                2   2.650250   1.835970        1.443515
13  Tomcat02       trace.from_IG01.duration_p95  2021-03-25 19:00:00+08:00                2   2.650250   1.835970        1.443515
14      MG02  trace.from_dockerB1.duration_mean  2021-03-25 19:00:00+08:00                2   0.112634   0.079330        1.419816
15  Tomcat04           trace.self.duration_mean  2021-03-25 19:00:00+08:00                2   0.076977   0.054759        1.405754
16      MG02   trace.from_dockerB1.duration_p95  2021-03-25 19:00:00+08:00                2   1.001100   0.714800        1.400532
17      MG01   trace.from_Tomcat03.duration_p95  2021-03-25 19:00:00+08:00                2   1.768500   1.293850        1.366851
18  Tomcat03         trace.to_MG01.duration_p95  2021-03-25 19:00:00+08:00                2   1.768500   1.293850        1.366851
19  Tomcat02            trace.self.duration_p95  2021-03-25 19:00:00+08:00                2   0.114250   0.084950        1.344909,                   timestamp cmdb_id                         trace_name     value
0 2021-03-25 19:00:00+08:00    MG01   trace.from_dockerA1.duration_p95  0.745900
1 2021-03-25 19:00:00+08:00    MG02  trace.from_dockerB2.duration_mean  0.114095
2 2021-03-25 19:00:00+08:00    MG02  trace.from_dockerA1.duration_mean  0.181889
3 2021-03-25 19:00:00+08:00    MG01           trace.self.duration_mean  0.172493
4 2021-03-25 19:00:00+08:00    MG01   trace.from_dockerB2.duration_p95  0.492000)```
```